# 02 · Dataset understanding

## Purpose

Inspect the actual active CSV schema, values, dimensions, target, identifiers, and available competition test/submission schemas.

## Method

Generate a data dictionary from observed CSV values and attach descriptions only for documented competition fields. Compare train/test feature schemas without evaluating unlabeled competition test rows.

## Interpretation

Use the displayed tables, figures, and saved metrics produced by this run. Do not fill this section with assumed outcomes. For every result, distinguish observed evidence from interpretation.

## Limitations

The competition test split has no labels. Semantic descriptions for undocumented columns remain explicitly unknown until confirmed from the source documentation.

## References

Competition data page: https://www.kaggle.com/competitions/retail-return-risk-modeling/data


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'returnwise').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from returnwise.research_state import load_state, save_state
state = load_state()
print('Project:', ROOT)
print('Notebook:', '02 · Dataset understanding')


Project: C:\Users\Lenovo\Documents\Who is back
Notebook: 02 · Dataset understanding


In [2]:
from returnwise.data.profile import data_dictionary
from returnwise.paths import KAGGLE_DIR
from returnwise.storage import load_active_dataset
import pandas as pd

frame, manifest = load_active_dataset()
if frame is None:
    raise RuntimeError('Select an imported or downloaded dataset in notebook 01 first.')
print('Source:', manifest)
print('Dimensions:', frame.shape)
print('Target present:', 'returned' in frame.columns)
print('Data types and unique values:')
display(pd.DataFrame({'dtype': frame.dtypes.astype(str), 'unique': frame.nunique(dropna=True), 'missing': frame.isna().sum()}))
print('Automatic data dictionary:')
dictionary = data_dictionary(frame)
display(dictionary)

test_path = KAGGLE_DIR / 'test.csv'
sample_path = KAGGLE_DIR / 'sample_submission.csv'
if manifest.get('source') == 'kaggle' and test_path.exists():
    competition_test = pd.read_csv(test_path)
    train_features = [c for c in frame.columns if c != 'returned']
    print('Competition test shape (labels are unavailable):', competition_test.shape)
    print('Train/test feature schema identical:', train_features == list(competition_test.columns))
    print('Columns only in active train:', sorted(set(train_features) - set(competition_test.columns)))
    print('Columns only in competition test:', sorted(set(competition_test.columns) - set(train_features)))
    if sample_path.exists():
        sample_submission = pd.read_csv(sample_path)
        print('Sample submission schema:', list(sample_submission.columns))
        display(sample_submission.head())
else:
    print('No competition test split is present; this is a manually imported dataset.')


Source: {'source': 'kaggle', 'file': 'data/kaggle/train.csv', 'rows': 200000, 'columns': ['customer_age', 'product_price', 'discount_percent', 'product_rating', 'past_purchase_count', 'past_return_rate', 'delivery_delay_days', 'session_length_minutes', 'num_product_views', 'device_type', 'product_category', 'shipping_method', 'payment_method', 'used_coupon', 'returned', 'order_id'], 'imported_at': '2026-11-15T06:04:58+00:00', 'dataset_version': '2ae5e87906198d2e', 'has_target': True}
Dimensions: (200000, 16)
Target present: True
Data types and unique values:


,dtype,unique,missing
customer_age,int64,78,0
product_price,float64,199800,0
discount_percent,float64,199907,0
product_rating,float64,199936,0
past_purchase_count,int64,26,0
past_return_rate,float64,199934,0
delivery_delay_days,float64,199947,0
session_length_minutes,float64,199921,0
num_product_views,int64,50,0
device_type,str,3,0


Automatic data dictionary:


,column,role,dtype,non_null,missing,missing_percent,unique_values,examples_from_file,description
0,customer_age,numeric feature,int64,200000,0,0.0,78,"25, 76, 40, 62",Customer age (competition description).
1,product_price,numeric feature,float64,200000,0,0.0,199800,"11.66613697698637, 31.37896692422914, 25.46060...",Product price at purchase (competition descrip...
2,discount_percent,numeric feature,float64,200000,0,0.0,199907,"43.55960142346256, 66.52603939172272, 20.76237...",Discount applied to the product (competition d...
3,product_rating,numeric feature,float64,200000,0,0.0,199936,"1.9335331058958345, 0.6902369777604651, 2.3476...",Average product rating (competition description).
4,past_purchase_count,numeric feature,int64,200000,0,0.0,26,"8, 9, 14, 10",Number of the customer's past purchases (compe...
5,past_return_rate,numeric feature,float64,200000,0,0.0,199934,"0.3314077354329958, 0.10520314818028009, 0.236...",Historical customer return rate (competition d...
6,delivery_delay_days,numeric feature,float64,200000,0,0.0,199947,"-0.08963734587823691, 3.63927684386212, 0.2940...",Delay against estimated delivery (competition ...
7,session_length_minutes,numeric feature,float64,200000,0,0.0,199921,"3.273029019390499, 130.51312301464336, 92.7809...",Duration of the browsing session in minutes (c...
8,num_product_views,numeric feature,int64,200000,0,0.0,50,"26, 8, -2, 7",Products viewed during the session (competitio...
9,device_type,categorical feature,str,200000,0,0.0,3,"tablet, mobile, desktop",Device category used for the session (competit...


Competition test shape (labels are unavailable): (50000, 15)
Train/test feature schema identical: True
Columns only in active train: []
Columns only in competition test: []
Sample submission schema: ['ID', 'returned']


,ID,returned
0,38683,0
1,64939,0
2,3954,0
3,120374,0
4,172861,0
